# Topic 3 — Local LLM SEO Diagnostics (Llama-3-8B-Instruct, QLoRA)

Self-contained Colab notebook for `topic-3-local-llm-seo-diagnostics/`, as
of 2026-09-13. See `docs/component-specs.md` for the full architecture —
this notebook implements both phases described there.

**Platform/model exception** (flagged in `docs/problem-statement-categorized.md`'s
run notes, not a silent substitution): the source asks for **Kaggle
Kernels**; this deploys on **Colab with a T4 GPU** instead — comparable
free-tier GPU capability, chosen for consistency with topics 1-2's
existing Colab workflow. Model: **Llama-3-8B-Instruct** (the other named
option, Gemma-2-9B-IT, was not used).

**Two phases:**
- **Phase A (offline, once)**: fine-tune Llama-3-8B-Instruct via QLoRA on
  12 gold-labeled training examples (`data/split_and_gold_labels.json`,
  full methodology in `docs/data-split-and-labels.md`). Produces a LoRA
  adapter.
- **Phase B (per held-out row)**: batch-diagnose the 8 held-out rows with
  *both* the base model and the fine-tuned model, at 3 temperatures each
  (a consistency test — B8), validate every output against a strict JSON
  schema, and render an HTML report + DataFrame.

**Honest scope note on verification**: unlike topics 1-2's notebooks, the
model-loading/training/generation cells below were **not executed in the
environment that built this notebook** — no GPU, no accepted Llama-3
license, no way to run them there. They follow standard, well-established
QLoRA/PEFT patterns, but you are the first one to actually run them
end to end. The JSON validation logic and data loading *were* tested
locally (see the cells that don't need `torch`) — everything that could be
verified without a GPU was.

**Before running — real prerequisites, not optional:**
1. A Hugging Face account with **Meta's Llama-3 license accepted**
   (visit the [Llama-3-8B-Instruct model page](https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct)
   and request/accept access — this can take a few minutes to a few hours
   to be approved).
2. An HF **access token** (Settings → Access Tokens → new token, read
   access is enough), added as a Colab secret named `HF_TOKEN`.
3. **Runtime → Change runtime type → T4 GPU** — this will not run on CPU
   in any reasonable time.
4. Expect the fine-tune + full Phase B batch to take **15-30 minutes**
   total on a T4 — mostly Phase B's 48 generation calls (8 rows × 2 model
   variants × 3 temperatures), not the fine-tune itself (12 examples, 3
   epochs is fast).

In [1]:
!pip install -q "transformers>=4.43.0" "peft>=0.12.0" "accelerate>=0.33.0" "bitsandbytes>=0.43.0" "pydantic>=2.0.0" "pandas>=2.0.0"


## 1. Credentials — Hugging Face token from Colab Secrets

Needed to download the gated Llama-3-8B-Instruct weights. Add `HF_TOKEN` via the 🔑 Secrets panel in the left sidebar (grant this notebook access when prompted), then run this cell.

In [2]:
import json
import re
from pathlib import Path
from typing import List

import pandas as pd
import torch
from google.colab import userdata
from huggingface_hub import login
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from pydantic import BaseModel, Field, field_validator
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except userdata.SecretNotFoundError:
    raise RuntimeError(
        "Colab secret 'HF_TOKEN' isn't set. Add it via the \N{KEY} Secrets "
        "panel in the left sidebar (a Hugging Face access token with read "
        "access), then re-run this cell."
    ) from None
except userdata.NotebookAccessError:
    raise RuntimeError(
        "Colab secret 'HF_TOKEN' exists but this notebook hasn't been "
        "granted access yet -- toggle notebook access on for it in the "
        "Secrets panel, then re-run this cell."
    ) from None

login(token=HF_TOKEN)
print("Logged in to Hugging Face. GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected -- Runtime -> Change runtime type -> T4 GPU, then re-run.")


Logged in to Hugging Face. GPU available: True
GPU: Tesla T4


## 2. Data — the stratified split + gold labels

Embedded verbatim from `data/split_and_gold_labels.json` (methodology in `docs/data-split-and-labels.md`) — not retyped.

In [3]:
SPLIT_DATA = json.loads('{\n  "score_scale": "0-100 (engineering default -- B6 doesn\'t specify a range; see docs/data-split-and-labels.md)",\n  "type_taxonomy": {\n    "A-legitimate": "Genuinely good content: accurate, appropriately hedged, no violations.",\n    "B-ymyl-overclaim": "YMYL guarantee/overclaim language (e.g. \\"guaranteed profit\\", \\"guaranteed approval\\").",\n    "C-planted-false-theory": "A deliberately planted, factually incorrect SEO theory.",\n    "D-keyword-stuffing": "Naked, unnatural keyword repetition with no informational content."\n  },\n  "train_ids": [\n    1,\n    2,\n    4,\n    5,\n    6,\n    7,\n    8,\n    9,\n    10,\n    16,\n    17,\n    18\n  ],\n  "held_out_ids": [\n    3,\n    11,\n    12,\n    13,\n    14,\n    15,\n    19,\n    20\n  ],\n  "train": [\n    {\n      "id": 1,\n      "category": "Casino",\n      "keyword": "娛樂城推薦",\n      "snippet": "挑選娛樂城時，首重資安與出金紀錄。建議觀察其是否有國際博弈執照，如菲律賓 PAGCOR，並從小額出金測試其速度。",\n      "type_tag": "A-legitimate",\n      "gold": {\n        "score": 88,\n        "reasons": [\n          "Cites a real, verifiable license authority (Philippines PAGCOR) and a concrete verification method (small withdrawal test) rather than a vague safety claim — an Expertise/Trustworthiness signal, not just an assertion.",\n          "No overclaiming, no guarantee language, no YMYL red flags."\n        ],\n        "actionable_advice": "Add brief context on why these two checks matter (e.g. what recourse a licensed operator\'s jurisdiction provides, a typical withdrawal-speed benchmark) to deepen the Expertise signal beyond a bare checklist."\n      }\n    },\n    {\n      "id": 2,\n      "category": "Casino",\n      "keyword": "娛樂城推薦",\n      "snippet": "2026最強娛樂城推薦！這裡保證 100% 獲利，每天登入送一萬，穩賺不賠，是您發家致富的唯一選擇，快來加入。",\n      "type_tag": "B-ymyl-overclaim",\n      "gold": {\n        "score": 8,\n        "reasons": [\n          "\\"保證100%獲利\\" / \\"穩賺不賠\\" (guaranteed 100% profit / can\'t lose) is a textbook false and misleading financial claim — gambling inherently carries risk, and this framing could cause real financial harm to a reader who believes it. Directly violates YMYL trustworthiness.",\n          "Zero substantive or verifiable information; pure promotional hyperbole with no checkable detail."\n        ],\n        "actionable_advice": "Remove all guarantee/certainty language entirely. Replace with realistic, risk-aware framing (e.g. explicit \\"entertainment, not an income source\\" disclosure) and substantive, checkable information in place of hype."\n      }\n    },\n    {\n      "id": 4,\n      "category": "Casino",\n      "keyword": "娛樂城出金",\n      "snippet": "如果您在搜尋娛樂城出金，請務必填寫 Meta Keywords 標籤，這在 2026 年是決定 Google 排名的最核心因素。",\n      "type_tag": "C-planted-false-theory",\n      "gold": {\n        "score": 10,\n        "reasons": [\n          "The Meta Keywords tag has not been a Google ranking factor since Google explicitly confirmed this publicly in 2009 — presenting it as \\"the core ranking factor in 2026\\" is a demonstrably false, outdated SEO theory, not a matter of opinion.",\n          "This is exactly the class of misinformation this audit exists to catch: a reader who acts on it wastes effort on an ineffective tactic while neglecting what actually affects ranking."\n        ],\n        "actionable_advice": "Remove this claim entirely. If discussing meta tags, redirect to the Meta Description (which affects click-through rate, not ranking directly) and name actual current ranking factors (content quality, E-E-A-T signals, page experience)."\n      }\n    },\n    {\n      "id": 5,\n      "category": "Casino",\n      "keyword": "娛樂城優惠",\n      "snippet": "根據我在各家娛樂城實測三個月的心得，首儲優惠雖然吸引人，但必須注意流水倍數限制，通常 15-20 倍才算合理。",\n      "type_tag": "A-legitimate",\n      "gold": {\n        "score": 90,\n        "reasons": [\n          "Explicit, specific first-hand experience claim (a named duration — three months of actual testing) paired with a concrete, checkable benchmark (15-20x wagering requirement) — exactly the kind of detail that\'s hard to fabricate and signals genuine Experience, not templated content.",\n          "Balanced tone: flags a real caveat (the wagering-multiple catch) rather than pure promotion."\n        ],\n        "actionable_advice": "Could name which specific platforms/promotions were tested (if disclosure permits) to make the experience claim independently verifiable."\n      }\n    },\n    {\n      "id": 6,\n      "category": "Mortgage",\n      "keyword": "房屋二胎利率",\n      "snippet": "辦理房屋二胎利率前，應先向原貸銀行詢問增貸可能。若信用正常，銀行利率通常落在 2%-5%，遠低於民間代書。",\n      "type_tag": "A-legitimate",\n      "gold": {\n        "score": 85,\n        "reasons": [\n          "Rate range (2%-5%) is specific and checkable, and is appropriately conditioned on \\"若信用正常\\" (if credit is normal) rather than presented as an unconditional guarantee — the correct YMYL disclosure pattern this dataset\'s violation rows (2, 7) get wrong.",\n          "Gives a genuinely useful, concrete first step (ask the original lender about a top-up loan before shopping elsewhere)."\n        ],\n        "actionable_advice": "Cite a source/date for the rate range, since mortgage rates move with policy and an undated figure ages poorly."\n      }\n    },\n    {\n      "id": 7,\n      "category": "Mortgage",\n      "keyword": "房屋二胎利率",\n      "snippet": "房屋二胎利率最低 0.1% 起！不看信用、不看財力，只要有房子保證過件，全台最低利，沒過件我們賠你錢。",\n      "type_tag": "B-ymyl-overclaim",\n      "gold": {\n        "score": 3,\n        "reasons": [\n          "Stacks multiple severe YMYL violations: an implausible rate (\\"0.1%起\\"), guaranteed-approval language regardless of creditworthiness (\\"不看信用、不看財力...保證過件\\"), and an unverifiable superlative (\\"全台最低利\\") — this is the same prohibited-language pattern (guaranteed approval / lowest-rate-nationwide) flagged as non-compliant in this project\'s own topic-2 writing manual.",\n          "Ignoring credit/financial capacity entirely combined with a dubious \\"refund if not approved\\" gimmick is a classic predatory-lending marketing pattern, not a legitimate offer description."\n        ],\n        "actionable_advice": "Remove all guarantee/superlative language. Replace with realistic, credit-conditional rate ranges and standard, verifiable application requirements — this content as written risks materially misleading a financially vulnerable reader."\n      }\n    },\n    {\n      "id": 8,\n      "category": "Mortgage",\n      "keyword": "房屋二胎申請",\n      "snippet": "申請房屋二胎需要準備身分證影本、土地建物所有權狀以及最近半年的房貸繳款紀錄，以利金融機構評估額度。",\n      "type_tag": "A-legitimate",\n      "gold": {\n        "score": 87,\n        "reasons": [\n          "Concrete, accurate, verifiable checklist of required application documents — directly useful and actionable.",\n          "No overclaiming, no YMYL red flags; purely factual/practical content."\n        ],\n        "actionable_advice": "Add where/how to obtain each document (e.g. which agency issues the ownership certificate) for added practical completeness."\n      }\n    },\n    {\n      "id": 9,\n      "category": "Mortgage",\n      "keyword": "房屋二胎申請",\n      "snippet": "在撰寫這篇文章時，我建議在 HTML 原始碼中將關鍵字字體顏色設為白色，隱藏在背景中，這樣可以騙過 Google 機器人。",\n      "type_tag": "C-planted-false-theory",\n      "gold": {\n        "score": 5,\n        "reasons": [\n          "Recommending white-on-white hidden text is not just outdated advice — it\'s a black-hat technique explicitly prohibited by Google\'s Webmaster Guidelines and can trigger manual penalties or de-indexing.",\n          "The claim that this \\"fools Google\'s crawler\\" is factually false: modern crawlers reliably detect hidden/cloaked text. This is actively harmful advice, not a neutral inaccuracy."\n        ],\n        "actionable_advice": "Remove this recommendation entirely. Replace with legitimate on-page guidance: natural keyword placement in genuinely visible content, proper heading structure."\n      }\n    },\n    {\n      "id": 10,\n      "category": "Casino",\n      "keyword": "娛樂城詐騙",\n      "snippet": "預防娛樂城詐騙，應避開標榜「保證獲利」或「老師帶牌」的群組。真實的娛樂城純屬娛樂，不可能有預測後台的技術。",\n      "type_tag": "A-legitimate",\n      "gold": {\n        "score": 89,\n        "reasons": [\n          "Correctly identifies \\"guaranteed profit\\" and \\"insider tipster groups\\" as scam red flags — accurate, protective YMYL guidance grounded in how these scams actually operate.",\n          "Explicitly frames gambling as entertainment rather than a prediction-based income strategy — responsible and factually accurate, the correct counter-framing to rows like 2 and 7."\n        ],\n        "actionable_advice": "Add concrete next steps for a reader who suspects they\'ve encountered such a scam (e.g. where to report it) to increase practical value."\n      }\n    },\n    {\n      "id": 16,\n      "category": "Mortgage",\n      "keyword": "房屋二胎申請",\n      "snippet": "房屋二胎房屋二胎房屋二胎。二胎房貸二胎房貸。我們提供最好的房屋二胎服務。歡迎洽詢房屋二胎。",\n      "type_tag": "D-keyword-stuffing",\n      "gold": {\n        "score": 5,\n        "reasons": [\n          "Keyword repeated six times across four short sentences with no natural language variation or informational content — the same violation pattern as row 3, in the Mortgage category.",\n          "Provides zero actionable information for a genuine reader; reads as spam, not content."\n        ],\n        "actionable_advice": "Rewrite around genuine informational value (specific loan terms, eligibility criteria) using natural keyword variants instead of literal repetition."\n      }\n    },\n    {\n      "id": 17,\n      "category": "Casino",\n      "keyword": "娛樂城詐騙",\n      "snippet": "最近出現一種偽裝成官方客服的詐騙手法，會要求玩家提供簡訊驗證碼。請記住，正規平台客服絕不會詢問您的驗證碼。",\n      "type_tag": "A-legitimate",\n      "gold": {\n        "score": 88,\n        "reasons": [\n          "Describes a specific, real, currently-relevant scam pattern (impersonating official support to phish an SMS verification code) rather than a generic warning.",\n          "Gives one clear, memorable, actionable rule (legitimate support never asks for your verification code) that a reader can actually apply."\n        ],\n        "actionable_advice": "Add what to do if a reader has already shared a code (e.g. immediate account-security steps) to make the warning more practically complete."\n      }\n    },\n    {\n      "id": 18,\n      "category": "Mortgage",\n      "keyword": "房屋二胎利率",\n      "snippet": "房屋二胎利率受央行升息政策影響，建議參考內政部最新的不動產資訊平台數據，以獲取最客觀的市場行情。",\n      "type_tag": "A-legitimate",\n      "gold": {\n        "score": 90,\n        "reasons": [\n          "Correctly attributes rate movement to a real macro factor (central bank policy) rather than asserting a specific number itself.",\n          "Directs readers to an official government data source (內政部 / Ministry of the Interior real-estate platform) for the actual figures — a strong, verifiable Authoritativeness pattern other rows in this dataset (especially 2, 7) fail to follow."\n        ],\n        "actionable_advice": "Name the specific platform/URL directly for immediate accessibility — the sourcing pattern itself is already exemplary."\n      }\n    }\n  ],\n  "held_out": [\n    {\n      "id": 3,\n      "category": "Casino",\n      "keyword": "娛樂城推薦",\n      "snippet": "娛樂城推薦娛樂城推薦。想要最好的娛樂城推薦嗎？我們有最棒的娛樂城推薦服務，娛樂城推薦就在這裡。",\n      "type_tag": "D-keyword-stuffing"\n    },\n    {\n      "id": 11,\n      "category": "Mortgage",\n      "keyword": "二胎房貸風險",\n      "snippet": "二胎房貸的風險在於清償順位。若發生法拍，二胎債權人僅能領取一胎清償後的剩餘價值，因此利率通常較高。",\n      "type_tag": "A-legitimate"\n    },\n    {\n      "id": 12,\n      "category": "Casino",\n      "keyword": "娛樂城推薦",\n      "snippet": "雖然這家娛樂城介面很華麗，但在 PTT 上的評價兩極，特別是客服回覆速度較慢，這是我實際註冊後最直觀的感受。",\n      "type_tag": "A-legitimate"\n    },\n    {\n      "id": 13,\n      "category": "Mortgage",\n      "keyword": "房屋二胎利率",\n      "snippet": "2026 年最有效的 SEO 技巧是將關鍵字放在 H1 標籤內重複 20 次，這能顯著提升網頁的權威性與排名。",\n      "type_tag": "C-planted-false-theory"\n    },\n    {\n      "id": 14,\n      "category": "Mortgage",\n      "keyword": "二胎房貸風險",\n      "snippet": "房貸二胎沒風險啦！反正房子在那裡，債務隨便還都可以，銀行不會隨便拍賣你的房子的，這點大可放心申請。",\n      "type_tag": "B-ymyl-overclaim"\n    },\n    {\n      "id": 15,\n      "category": "Casino",\n      "keyword": "娛樂城優惠",\n      "snippet": "娛樂城優惠領取教學：1.註冊帳號 2.完成實名認證 3.聯繫客服領取體驗金。注意：多重帳號領取會被封號。",\n      "type_tag": "A-legitimate"\n    },\n    {\n      "id": 19,\n      "category": "Casino",\n      "keyword": "娛樂城推薦",\n      "snippet": "在 Google RankBrain 演算法中，內容的字數多寡是唯一的排名標準。只要字數超過一萬字，不論內容品質，一定能排到第一頁。",\n      "type_tag": "C-planted-false-theory"\n    },\n    {\n      "id": 20,\n      "category": "Mortgage",\n      "keyword": "二胎房貸風險",\n      "snippet": "民法第 873 條規定，抵押權人於債權已屆清償期而未受清償者，得聲請法院拍賣抵押物。這是申請二胎前必須明白的法律後果。",\n      "type_tag": "A-legitimate"\n    }\n  ]\n}')

TRAIN = SPLIT_DATA["train"]           # 12 rows, each with a "gold" label
HELD_OUT = SPLIT_DATA["held_out"]     # 8 rows, no label -- these get diagnosed

print(f"Train: {len(TRAIN)} rows | Held-out: {len(HELD_OUT)} rows")
print("Held-out IDs:", [r["id"] for r in HELD_OUT])
print("Score scale:", SPLIT_DATA["score_scale"])


Train: 12 rows | Held-out: 8 rows
Held-out IDs: [3, 11, 12, 13, 14, 15, 19, 20]
Score scale: 0-100 (engineering default -- B6 doesn't specify a range; see docs/data-split-and-labels.md)


## 3. The diagnostic audit prompt

Embedded verbatim from `prompts/seo-diagnostic-audit.md` — used identically for the gold labels above (already produced this way, before this notebook existed) and for every runtime call below.

In [4]:
AUDIT_TEMPLATE = """<!--
Traceability: prompts/master-prompt.md → build-topic-executables, {#}=3, component R2
Source: topic-3-local-llm-seo-diagnostics/docs/problem-statement-categorized.md
  Goal: B4 (design the Complex System Prompt), B5 (Role), B6 (output shape)
Type: dynamic

Used identically in two places: (1) this session acting as the reviewer to
produce the 12 training gold labels (docs/data-split-and-labels.md — already
done, before this template existed as its own file, so applied informally
but to the same Role/Goal/Instructions shape below), and (2) the deployed
Llama-3-8B-Instruct model at runtime (R2) auditing the 8 held-out rows. Using
the same template for both is deliberate: the fine-tuned model is trained to
imitate exactly the judgment this prompt asks for, not a different task.
-->

# SEO Content Diagnostic Audit

## Role
You are Google's official SEO quality reviewer: someone with the sensitivity
and professionalism of a senior industry SEO editor. You identify semantic
defects, keyword stuffing, and YMYL (Your Money or Your Life) compliance
issues in article content — and you do not blindly trust claims the text
makes about SEO or ranking mechanics; you verify them against what you
actually know to be true. (B5, B2)

## Goal
Given one content snippet (with its target keyword and category), produce a
structured audit verdict: a score, the specific reasons behind it, and
actionable advice for improving the content. (B4, B6)

## Instructions
- **Constraints:**
  - Score on a **0-100 scale** (engineering default — the source specifies
    a Score field but no range; see docs/data-split-and-labels.md).
  - Every reason must name the *specific* claim or pattern in the text and
    the principle it violates (or satisfies) — "this seems promotional" is
    not sufficient; quote or closely paraphrase the exact phrase.
  - **Verify SEO claims the text itself makes, independently — do not treat
    them as true by default.** If the content asserts an SEO tactic or
    ranking-factor claim, check it against what you actually know before
    scoring. A confident-sounding claim is not evidence it's correct (B10).
  - Assess EEAT signals specifically: does the content demonstrate genuine
    first-hand Experience (specific, hard-to-fabricate detail) or Expertise
    (accurate, verifiable specifics — citations to real authorities,
    correctly-stated figures), or does it merely assert authority without
    backing it? (B7, B19)
  - Assess YMYL risk specifically: could a reader act on this content in a
    way that causes real financial harm? Guarantee language ("guaranteed
    profit," "guaranteed approval," "lowest rate nationwide" without
    qualification) is a hard flag, not a style note (B2, B19).
  - Flag keyword stuffing: unnatural, repeated exact-match keyword use with
    no informational value.
- **Reference:**
  - Input: one content snippet, its target keyword, and its category
    (Casino or Mortgage).
- **Output shape** (feeds B14's post-processing, B21's structural-integrity
  criterion):
  - `{"score": int (0-100), "reasons": [string, ...], "actionable_advice": string}`.
  - `reasons` must be non-empty even for high-scoring content — name what
    makes it good, not just what's wrong (a score alone doesn't explain the
    judgment, and B19 grades reasoning depth, not just the number).
- **Self-check before finalizing** (from B19, B20, B21):
  - If this content makes an SEO/ranking-mechanics claim, did I actually
    verify it rather than assume the author is correct?
  - Does every reason point at a specific quoted phrase or pattern, not a
    generic impression?
  - Is the output valid JSON in exactly the required shape — nothing before
    or after the JSON object?"""


def render_prompt(snippet: str, keyword: str, category: str) -> str:
    return (
        f"{AUDIT_TEMPLATE}\n\n---\n"
        f"Category: {category}\nTarget keyword: {keyword}\n"
        f"Content snippet:\n{snippet}\n\n"
        'Respond with ONLY a JSON object of this exact shape, nothing else -- '
        'no markdown code fence, no preamble, no explanation outside the JSON: '
        '{"score": <int 0-100>, "reasons": ["...", "..."], "actionable_advice": "..."}'
    )


## 4. R3 — JSON validation gate

A local 8B model is meaningfully more likely to wrap its answer in a markdown code fence or add a preamble than the hosted models topics 1-2 used, so `extract_json` is more defensive than a bare `json.loads` — it strips a fenced block if present, then finds the first balanced `{...}` substring by bracket-counting, rather than assuming the whole response is clean JSON.

In [5]:
class AuditVerdict(BaseModel):
    """Output of R2. Source: B6 (+B21 -- 100% structural compliance is graded directly)."""

    score: int = Field(ge=0, le=100)
    reasons: List[str]
    actionable_advice: str

    @field_validator("reasons")
    @classmethod
    def _reasons_non_empty(cls, v: List[str]) -> List[str]:
        if not v:
            raise ValueError("reasons must be non-empty -- a score alone doesn't explain the judgment (B19)")
        return v

    @field_validator("actionable_advice")
    @classmethod
    def _advice_non_empty(cls, v: str) -> str:
        if not v.strip():
            raise ValueError("actionable_advice must be non-empty")
        return v


_FENCE_RE = re.compile(r"```(?:json)?\s*(.*?)\s*```", re.DOTALL)


def extract_json(raw: str) -> str:
    """Defensive extraction, not a bare json.loads -- see markdown above."""
    fenced = _FENCE_RE.search(raw)
    text = fenced.group(1) if fenced else raw

    start = text.find("{")
    if start == -1:
        raise ValueError(f"No JSON object found in model output: {raw!r}")
    depth = 0
    for i, ch in enumerate(text[start:], start=start):
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start:i + 1]
    raise ValueError(f"Unbalanced braces in model output: {raw!r}")


def parse_and_validate(raw: str) -> dict:
    """Returns {"ok": True, "verdict": AuditVerdict} or {"ok": False, "error": str, "raw": raw}.
    Never raises -- a validation failure is a recorded result (R3's error-handling
    contract), not something that aborts the batch."""
    try:
        obj = json.loads(extract_json(raw))
        verdict = AuditVerdict.model_validate(obj)
        return {"ok": True, "verdict": verdict}
    except Exception as exc:
        return {"ok": False, "error": str(exc), "raw": raw}


### Sanity check — verify the parsing logic against known-good and known-bad shapes

Doesn't need the model loaded; runs now so a bug here is caught before it's buried under 48 real generation calls later.

In [6]:
_test_cases = [
    ('{"score": 75, "reasons": ["r1"], "actionable_advice": "a1"}', True),
    ('```json\n{"score": 40, "reasons": ["r1", "r2"], "actionable_advice": "a1"}\n```', True),
    ('Sure, here is the audit:\n{"score": 60, "reasons": ["r1"], "actionable_advice": "a1"}\nHope that helps!', True),
    ('{"score": 150, "reasons": ["r1"], "actionable_advice": "a1"}', False),  # out of range
    ('{"score": 50, "reasons": [], "actionable_advice": "a1"}', False),       # empty reasons
    ('not json at all', False),
]
for raw, should_pass in _test_cases:
    result = parse_and_validate(raw)
    status = "PASS" if result["ok"] == should_pass else "FAIL"
    print(f"[{status}] expected ok={should_pass}, got ok={result['ok']!r} for: {raw[:50]!r}")


[PASS] expected ok=True, got ok=True for: '{"score": 75, "reasons": ["r1"], "actionable_advic'
[PASS] expected ok=True, got ok=True for: '```json\n{"score": 40, "reasons": ["r1", "r2"], "ac'
[PASS] expected ok=True, got ok=True for: 'Sure, here is the audit:\n{"score": 60, "reasons": '
[PASS] expected ok=False, got ok=False for: '{"score": 150, "reasons": ["r1"], "actionable_advi'
[PASS] expected ok=False, got ok=False for: '{"score": 50, "reasons": [], "actionable_advice": '
[PASS] expected ok=False, got ok=False for: 'not json at all'


## 5. Load Llama-3-8B-Instruct in 4-bit

Loaded once; Phase B toggles between "base" and "fine-tuned" behavior on this *same* loaded model via PEFT's adapter enable/disable, rather than loading the 8B model twice into GPU memory.

In [7]:
MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
SEED = 42  # fixed throughout -- data split, LoRA init, training shuffling (component-specs.md run notes)

torch.manual_seed(SEED)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
)
print("Base model loaded.")


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Base model loaded.


## 6. R2 — diagnostic inference call

One call, one row, one temperature. Uses Llama-3-Instruct's chat template (required for instruction-following behavior -- raw text completion without it produces much weaker results on an Instruct-tuned model).

In [8]:
def generate_diagnosis(model, tokenizer, prompt: str, temperature: float, max_new_tokens: int = 512) -> str:
    messages = [{"role": "user", "content": prompt}]
    # return_dict=True explicitly, then pull input_ids/attention_mask out by
    # name -- apply_chat_template's return type without return_dict isn't
    # consistent across transformers versions (a bare tensor in some, a
    # BatchEncoding dict-like in others), and only pinning a loose
    # requirements floor means whichever version Colab actually installs is
    # out of our control. This is the one unambiguous way to get it either way.
    encoded = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    input_ids = encoded["input_ids"]

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            attention_mask=encoded["attention_mask"],
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=max(temperature, 0.01),  # 0.0 is invalid for some samplers; floor it
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated = output_ids[0][input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


### Baseline sanity check — one real call before committing to the fine-tune

Verifies the model-loading + generation + parsing plumbing actually works end to end, cheaply, before the longer fine-tuning step.

In [9]:
_sample = HELD_OUT[0]
_prompt = render_prompt(_sample["snippet"], _sample["keyword"], _sample["category"])
_raw = generate_diagnosis(base_model, tokenizer, _prompt, temperature=0.7)
print("=== raw output ===")
print(_raw)
print("\n=== parsed ===")
print(parse_and_validate(_raw))


[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


=== raw output ===
{
  "score": 20,
  "reasons": [
    "The content is overly promotional and fails to provide any valuable information about the recommended entertainment city.",
    "The use of exact-match keywords ('娛樂城推薦') is unnatural and may be considered keyword stuffing."
  ],
  "actionable_advice": "Provide more detailed and informative content about the recommended entertainment city, and avoid using exact-match keywords. Instead, use natural language and focus on providing value to the reader."
}

=== parsed ===
{'ok': True, 'verdict': AuditVerdict(score=20, reasons=['The content is overly promotional and fails to provide any valuable information about the recommended entertainment city.', "The use of exact-match keywords ('娛樂城推薦') is unnatural and may be considered keyword stuffing."], actionable_advice='Provide more detailed and informative content about the recommended entertainment city, and avoid using exact-match keywords. Instead, use natural language and focus on pro

## 7. T3 — QLoRA fine-tune on the 12 gold-labeled training rows

**Real overfitting risk, named upfront (see `docs/component-specs.md`'s run notes)**: 12 examples is a small fine-tuning set. The base-vs-fine-tuned comparison in Phase B is specifically designed to surface this if it happens -- a fine-tuned model that got *worse* on held-out rows is a real, reportable finding for the Reflection (B18), not a bug to hide.

Hyperparameters (rank 16, alpha 32, lr 2e-4, 3 epochs, batch size 1 with gradient accumulation 4) are flagged engineering defaults, not sourced —
see `docs/component-specs.md`'s run notes.

**A T4 has 15GB of VRAM, and an 8B model's training forward/backward pass does not fit in that without gradient checkpointing** — the cell below calls `prepare_model_for_kbit_training(..., use_gradient_checkpointing=True)` before attaching LoRA; skip it and this reliably OOMs partway through the first epoch, not at load time, which is what makes it a confusing failure to hit blind.

In [10]:
# prepare_model_for_kbit_training does three things a 4-bit base model needs
# before LoRA training that get_peft_model alone does NOT do: casts norm
# layers to fp32 for training stability, calls enable_input_require_grads()
# so gradients still flow into the LoRA adapters through frozen, checkpointed
# input embeddings, and (via use_gradient_checkpointing=True) turns on
# activation checkpointing -- without it, backward stores full activations
# for all 32 layers of an 8B model and reliably OOMs a 15GB T4. Skipping this
# step is the single most common cause of QLoRA OOM on a T4; it is not
# optional the way it might look from a from_pretrained + get_peft_model
# example alone.
base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(base_model, lora_config)
model.config.use_cache = False  # incompatible with gradient checkpointing; re-enabled after training, below
model.print_trainable_parameters()


def build_training_example(row: dict) -> dict:
    """Formats one training row as (input_ids, labels) with the prompt
    tokens masked out (-100) so loss is only computed on the gold JSON
    completion -- the standard SFT label-masking pattern, not optional."""
    prompt = render_prompt(row["snippet"], row["keyword"], row["category"])
    target = json.dumps(row["gold"], ensure_ascii=False)

    # return_dict=True + pull tensors out by name -- see generate_diagnosis's
    # comment above for why (apply_chat_template's bare return type isn't
    # consistent across transformers versions).
    prompt_encoded = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}], add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    )
    full_encoded = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}, {"role": "assistant", "content": target}],
        return_tensors="pt", return_dict=True,
    )
    prompt_ids = prompt_encoded["input_ids"]
    full_ids = full_encoded["input_ids"]

    labels = full_ids.clone()
    labels[:, : prompt_ids.shape[1]] = -100
    return {"input_ids": full_ids, "attention_mask": full_encoded["attention_mask"], "labels": labels}


training_examples = [build_training_example(row) for row in TRAIN]
print(f"Built {len(training_examples)} training examples.")


trainable params: 13,631,488 || all params: 8,043,892,736 || trainable%: 0.1695
Built 12 training examples.


In [11]:
EPOCHS = 3
LEARNING_RATE = 2e-4
GRAD_ACCUM_STEPS = 4

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
model.train()

torch.manual_seed(SEED)
step = 0
for epoch in range(EPOCHS):
    perm = torch.randperm(len(training_examples))  # fixed seed above makes this reproducible
    epoch_loss = 0.0
    optimizer.zero_grad()
    for i, idx in enumerate(perm.tolist()):
        ex = training_examples[idx]
        out = model(
            input_ids=ex["input_ids"].to(model.device),
            attention_mask=ex["attention_mask"].to(model.device),
            labels=ex["labels"].to(model.device),
        )
        loss = out.loss / GRAD_ACCUM_STEPS
        loss.backward()
        epoch_loss += out.loss.item()
        step += 1
        if step % GRAD_ACCUM_STEPS == 0:
            optimizer.step()
            optimizer.zero_grad()
    print(f"epoch {epoch + 1}/{EPOCHS} -- mean loss: {epoch_loss / len(training_examples):.4f}")

optimizer.step()  # flush any remaining accumulated gradients
optimizer.zero_grad()
model.eval()
model.config.use_cache = True  # re-enable for fast generation in Phase B (was off for gradient checkpointing)
torch.cuda.empty_cache()

ADAPTER_DIR = "llama3-seo-audit-adapter"
model.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to ./{ADAPTER_DIR}")


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


epoch 1/3 -- mean loss: 2.5128
epoch 2/3 -- mean loss: 1.9874
epoch 3/3 -- mean loss: 1.6200
Adapter saved to ./llama3-seo-audit-adapter


## 8. Phase B — R1 + R4: diagnose the 8 held-out rows, base vs. fine-tuned, 3 temperatures each

48 total generation calls (8 rows × 2 model variants × 3 temperatures) — this is the slow cell, expect several minutes on a T4. Base-variant calls use `model.disable_adapter()` (PEFT's context manager) to temporarily run the *unmodified* base model on the same loaded weights, rather than loading the 8B model a second time.

In [12]:
TEMPERATURES = [0.1, 0.7, 1.0]  # engineering default, see component-specs.md run notes

results = []
for row in HELD_OUT:
    prompt = render_prompt(row["snippet"], row["keyword"], row["category"])
    for variant in ["base", "finetuned"]:
        for temp in TEMPERATURES:
            if variant == "base":
                with model.disable_adapter():
                    raw = generate_diagnosis(model, tokenizer, prompt, temperature=temp)
            else:
                raw = generate_diagnosis(model, tokenizer, prompt, temperature=temp)

            parsed = parse_and_validate(raw)
            results.append({
                "id": row["id"],
                "category": row["category"],
                "type_tag": row["type_tag"],
                "keyword": row["keyword"],
                "snippet": row["snippet"],
                "variant": variant,
                "temperature": temp,
                "raw": raw,
                **({"score": parsed["verdict"].score,
                    "reasons": parsed["verdict"].reasons,
                    "actionable_advice": parsed["verdict"].actionable_advice,
                    "valid_json": True}
                   if parsed["ok"] else
                   {"score": None, "reasons": None, "actionable_advice": None,
                    "valid_json": False, "validation_error": parsed["error"]}),
            })
    print(f"row {row['id']} done ({row['type_tag']})")

print(f"\nTotal results: {len(results)} (expected {len(HELD_OUT)} x 2 x {len(TEMPERATURES)} = {len(HELD_OUT) * 2 * len(TEMPERATURES)})")


[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

row 3 done (D-keyword-stuffing)


[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

row 11 done (A-legitimate)


[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

...

row 15 done (A-legitimate)


[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

row 19 done (C-planted-false-theory)


[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

row 20 done (A-legitimate)

Total results: 48 (expected 8 x 2 x 3 = 48)


## 9. R5 — post-processing: DataFrame + HTML report

Rows that failed JSON validation are shown explicitly as failures (R3's error-handling contract), not dropped — a local model's actual JSON compliance rate is itself B21 evidence, not something to hide.

In [13]:
df = pd.DataFrame(results)
print(f"JSON validity rate: {df['valid_json'].mean():.1%}  ({df['valid_json'].sum()}/{len(df)})")
df.to_csv("topic3_results.csv", index=False, encoding="utf-8-sig")
df


JSON validity rate: 89.6%  (43/48)


,id,category,type_tag,keyword,snippet,variant,temperature,raw,score,reasons,actionable_advice,valid_json,validation_error
0,3,Casino,D-keyword-stuffing,娛樂城推薦,娛樂城推薦娛樂城推薦。想要最好的娛樂城推薦嗎？我們有最棒的娛樂城推薦服務，娛樂城推薦就在這裡。,base,0.1,"{\n""score"": 20,\n""reasons"": [\n""Keyword stuffi...",20.0,[Keyword stuffing: repeated exact-match keywor...,Improve the content by providing specific exam...,True,NaN
1,3,Casino,D-keyword-stuffing,娛樂城推薦,娛樂城推薦娛樂城推薦。想要最好的娛樂城推薦嗎？我們有最棒的娛樂城推薦服務，娛樂城推薦就在這裡。,base,0.7,"{\n ""score"": 20,\n ""reasons"": [\n ""Keywor...",20.0,[Keyword stuffing: repeated exact-match keywor...,"Rewrite the content to provide more specific, ...",True,NaN
2,3,Casino,D-keyword-stuffing,娛樂城推薦,娛樂城推薦娛樂城推薦。想要最好的娛樂城推薦嗎？我們有最棒的娛樂城推薦服務，娛樂城推薦就在這裡。,base,1.0,"{\n ""score"": 20,\n ""reasons"": [\n ""Keywor...",20.0,[Keyword stuffing: the content contains repeat...,"Provide specific, verifiable details about the...",True,NaN
3,3,Casino,D-keyword-stuffing,娛樂城推薦,娛樂城推薦娛樂城推薦。想要最好的娛樂城推薦嗎？我們有最棒的娛樂城推薦服務，娛樂城推薦就在這裡。,finetuned,0.1,"{""score"": 0, ""reasons"": [""Keyword-stuffed exac...",0.0,[Keyword-stuffed exact-match keyword repetitio...,Rewrite as a natural sentence with a single ke...,True,NaN
4,3,Casino,D-keyword-stuffing,娛樂城推薦,娛樂城推薦娛樂城推薦。想要最好的娛樂城推薦嗎？我們有最棒的娛樂城推薦服務，娛樂城推薦就在這裡。,finetuned,0.7,"{""score"": 20, ""reasons"": [""Keyword-stuffed exa...",20.0,[Keyword-stuffed exact-match keyword use with ...,Replace with a genuine recommendation list (un...,True,NaN
5,3,Casino,D-keyword-stuffing,娛樂城推薦,娛樂城推薦娛樂城推薦。想要最好的娛樂城推薦嗎？我們有最棒的娛樂城推薦服務，娛樂城推薦就在這裡。,finetuned,1.0,"{""score"": 10, ""reasons"": [""Overly explicit, re...",10.0,"[Overly explicit, repeated, keyword-stuffed ex...",Replace with an actual recommendation (a real ...,True,NaN
6,11,Mortgage,A-legitimate,二胎房貸風險,二胎房貸的風險在於清償順位。若發生法拍，二胎債權人僅能領取一胎清償後的剩餘價值，因此利率通常較高。,base,0.1,"{\n""score"": 60,\n""reasons"": [\n""Claim about ra...",60.0,[Claim about ranking mechanics: '利率通常較高' is no...,"To improve the content, provide specific examp...",True,NaN
7,11,Mortgage,A-legitimate,二胎房貸風險,二胎房貸的風險在於清償順位。若發生法拍，二胎債權人僅能領取一胎清償後的剩餘價值，因此利率通常較高。,base,0.7,"{\n""score"": 60,\n""reasons"": [\n""二胎房貸的風險在於清償順位""...",60.0,"[二胎房貸的風險在於清償順位, the content makes an SEO claim...","Provide specific, verifiable details to demons...",True,NaN
8,11,Mortgage,A-legitimate,二胎房貸風險,二胎房貸的風險在於清償順位。若發生法拍，二胎債權人僅能領取一胎清償後的剩餘價值，因此利率通常較高。,base,1.0,"{\n""score"": 60,\n""reasons"": [\n""The content do...",60.0,[The content does not provide a clear definiti...,"To improve the content, provide a clear and co...",True,NaN
9,11,Mortgage,A-legitimate,二胎房貸風險,二胎房貸的風險在於清償順位。若發生法拍，二胎債權人僅能領取一胎清償後的剩餘價值，因此利率通常較高。,finetuned,0.1,"{""score"": 95, ""reasons"": [""Specific, accurate,...",95.0,"[Specific, accurate, verifiable detail (the co...",Add a concrete example (e.g. a specific loan s...,True,NaN


In [14]:
def _fmt_verdict(r) -> str:
    if not r["valid_json"]:
        return f'<div style="color:#a8412e;">VALIDATION FAILED: {r["validation_error"]}</div>'
    reasons_html = "".join(f"<li>{x}</li>" for x in r["reasons"])
    return (
        f'<b>Score: {r["score"]}/100</b>'
        f'<ul>{reasons_html}</ul>'
        f'<i>Advice: {r["actionable_advice"]}</i>'
    )


rows_html = []
for row in HELD_OUT:
    row_results = [r for r in results if r["id"] == row["id"]]
    variant_blocks = []
    for variant in ["base", "finetuned"]:
        temp_blocks = "".join(
            f'<div style="margin-bottom:8px;"><b>T={r["temperature"]}</b>: {_fmt_verdict(r)}</div>'
            for r in row_results if r["variant"] == variant
        )
        variant_blocks.append(f'<div style="flex:1;"><h4>{variant}</h4>{temp_blocks}</div>')
    rows_html.append(
        f'<div style="border:1px solid #ccc; margin:12px 0; padding:12px;">'
        f'<h3>Row {row["id"]} ({row["type_tag"]}) -- {row["category"]} / {row["keyword"]}</h3>'
        f'<p><i>{row["snippet"]}</i></p>'
        f'<div style="display:flex; gap:16px;">{"".join(variant_blocks)}</div>'
        f'</div>'
    )

html_report = f"<html><body>{''.join(rows_html)}</body></html>"
Path("topic3_report.html").write_text(html_report, encoding="utf-8")

from IPython.display import HTML, display
display(HTML(html_report))


## 10. Next steps

- Download `topic3_results.csv` and `topic3_report.html` (and the `llama3-seo-audit-adapter/` folder if you want to keep the fine-tuned weights) before the Colab runtime recycles — all three are local to this session only.
- The CSV/report above are the raw material for **B16** (Audit Report) — write up *why* the model's judgments were accurate or biased, specifically comparing base vs. fine-tuned on rows 13/19 (the held-out planted false theories) and on the consistency spread across temperatures.
- **B18** (Reflection) should draw on the actual JSON-validity rate and the base-vs-fine-tuned comparison observed above, not a generic writeup — including if the fine-tune helped only marginally or not at all, given 12 training examples is a small set.